# Small steps on a small scale: Integrating low-voltage DNs in ASSUME

## Notebook 1: Units as implemented in ASSUME and their ability to participate on a lokal market.

This notebook displays LV units as implemented in ASSUME and their ability to participate on a lokal market.

Deliverables:
1. Define a small test distribution network.
2. Define a local EOM market, where units can trade.
3. Subsequently add BASE, PV, HP, BSS and EV to the market.
4. Display resulting trading activity and ask yourself, if transactions make sense.

In [1]:
from pathlib import Path

import pandas as pd

from datetime import datetime

from assume import World
from assume.common.forecasts import NaiveForecast


In [2]:
start = datetime(2023, 1, 13, hour=0)
end = datetime(2023, 1, 13, hour=4)
temporal_resolution = "60min"
index = pd.date_range(start, end, freq=temporal_resolution)
n_steps = len(index)

# Create ASSUME World.
db_uri = "sqlite:///local_db/assume_db.db"
world = World(database_uri=db_uri)
world.setup(
    start=start,
    end=end,
    save_frequency_hours=1,
    simulation_id="reproduce_issue",
    index=index)

world.add_unit_operator(f"demand_operator")

print(f"{"unit_id":<25}{"volume.sum()":>10}")

for unit_id in ["demand_unit_0", "household_0", "schokolademandelbrot_0", "suisse_availability"]:

    demand_forecast = NaiveForecast(index, demand=100)

    world.add_unit(
        id=unit_id,
        unit_type="demand",
        unit_operator_id=f"demand_operator",
        unit_params={
            "min_power": 0,
            "max_power": 1000,
            "bidding_strategies": {"EOM": "naive_eom"},
            "technology": "demand"},
    forecaster=demand_forecast)

    volume = world.unit_operators["demand_operator"].units[unit_id].volume
    print(f"{unit_id:<25}{volume.sum():>10}")

world.run()

INFO:assume.world:Connected to the database
INFO:assume.world:Learning Strategies are not available. Check that you have torch installed.
unit_id                  volume.sum()
demand_unit_0                -500.0
household_0                  -500.0
schokolademandelbrot_0       -500.0
suisse_availability          -500.0


reproduce_issue 2023-01-13 04:00:00: : 14401.0it [00:00, 766116.69it/s]
